# Project DeepLure: Color-Invariant Saree Design Recognition
## An End-to-End Metric Learning System for Textile Retrieval and Verification
### Challenge: 'Face Recognition for Saree Designs' — Identifying underlying motif structures regardless of color palette.

---
### Cell 1: Environment Setup & Dependencies


In [ ]:
# Cell 1: Install dependencies
!pip install -q torch torchvision numpy pandas scikit-learn matplotlib seaborn pillow tqdm
!pip install -q faiss-cpu || true
print('All core dependencies installed successfully.')


### Cell 2: Imports, Global Configuration & Seeding


In [ ]:
# Cell 2: Imports & Configuration
import os
import sys
import math
import time
import random
import logging
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional, Any

import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageDraw, ImageEnhance, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, precision_recall_fscore_support, accuracy_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
import torchvision.transforms as T
import torchvision.models as models

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

@dataclass
class NotebookConfig:
    SEED: int = 42
    DEVICE: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    IMAGE_SIZE: int = 224
    BATCH_SIZE: int = 32
    EPOCHS: int = 15
    LEARNING_RATE: float = 1e-4
    BACKBONE_LR_RATIO: float = 0.1
    WEIGHT_DECAY: float = 1e-4
    EMBEDDING_DIM: int = 256
    P_CLASSES: int = 8
    K_SAMPLES: int = 4
    TEMPERATURE: float = 0.07
    TRIPLET_MARGIN: float = 0.3
    BACKBONE: str = 'convnext_tiny'
    BASELINE_BACKBONE: str = 'resnet50'
    EARLY_STOPPING_PATIENCE: int = 4
    DATA_ROOT: Path = Path('/kaggle/input')
    OUTPUT_DIR: Path = Path('/kaggle/working/outputs')

cfg = NotebookConfig()
seed_everything(cfg.SEED)
print(f'Initialized DeepLure Framework on Device: {cfg.DEVICE} | Backbone: {cfg.BACKBONE}')


### Cell 3: Dataset Paths Configuration (DeepLure & Kaggle Indian Saree Patterns)


In [ ]:
# Cell 3: Dataset Paths
KAGGLE_DATASET_DIR = Path('/kaggle/input/indian-saree-patterns')
DEEPLURE_DATASET_DIR = Path('/kaggle/input/deeplure-saree-corpus')
SYNTHETIC_DATASET_DIR = Path('/kaggle/working/synthetic_saree_data')

cfg.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Working output directory: {cfg.OUTPUT_DIR}')


### Cell 4: Dataset Inspection & Corpus Ingestion (Supports DeepLure, Kaggle & Synthetic)


In [ ]:
# Cell 4: Dataset Inspection & Data Ingestion
class SareeItem:
    def __init__(self, image_path: str, design_id: str, colorway_id: str = 'unknown'):
        self.image_path = str(image_path)
        self.design_id = str(design_id)
        self.colorway_id = str(colorway_id)
    def __repr__(self):
        return f'{self.design_id}_{self.colorway_id}'

def generate_synthetic_saree_corpus(output_dir, num_designs=24, colorways_per_design=5, image_size=256, seed=42):
    rng = random.Random(seed)
    np_rng = np.random.default_rng(seed)
    root = Path(output_dir)
    root.mkdir(parents=True, exist_ok=True)
    PALETTES = [
        {'name': 'crimson_gold', 'bg': (140, 20, 30), 'primary': (230, 190, 60), 'secondary': (200, 50, 60)},
        {'name': 'royal_blue', 'bg': (20, 45, 120), 'primary': (220, 220, 240), 'secondary': (50, 100, 200)},
        {'name': 'emerald_green', 'bg': (20, 100, 50), 'primary': (240, 220, 150), 'secondary': (60, 160, 90)},
        {'name': 'saffron_orange', 'bg': (210, 100, 20), 'primary': (255, 240, 180), 'secondary': (160, 50, 10)},
        {'name': 'deep_magenta', 'bg': (130, 20, 100), 'primary': (245, 210, 230), 'secondary': (180, 50, 150)},
        {'name': 'midnight_teal', 'bg': (15, 80, 90), 'primary': (210, 240, 230), 'secondary': (40, 130, 140)},
        {'name': 'golden_yellow', 'bg': (200, 160, 20), 'primary': (60, 30, 20), 'secondary': (240, 210, 80)},
        {'name': 'charcoal_silver', 'bg': (45, 45, 50), 'primary': (220, 220, 225), 'secondary': (120, 120, 130)},
    ]
    items = []
    for d_idx in range(num_designs):
        design_id = f'design_{d_idx:03d}'
        design_dir = root / design_id
        design_dir.mkdir(parents=True, exist_ok=True)
        motif_type = d_idx % 4
        element_size = 18 + (d_idx % 5) * 4
        selected_palettes = rng.sample(PALETTES, min(colorways_per_design, len(PALETTES)))
        for c_idx, pal in enumerate(selected_palettes):
            colorway_name = pal['name']
            img = Image.new('RGB', (image_size, image_size), pal['bg'])
            draw = ImageDraw.Draw(img)
            border_h = 35
            draw.rectangle([0, image_size - border_h, image_size, image_size], fill=pal['primary'])
            draw.line([0, image_size - border_h, image_size, image_size - border_h], fill=pal['secondary'], width=3)
            for bx in range(0, image_size, 16):
                draw.polygon([(bx, image_size), (bx + 8, image_size - border_h + 5), (bx + 16, image_size)], fill=pal['secondary'])
            step = max(32, image_size // 4)
            for y in range(step // 2, image_size - border_h - 10, step):
                for x in range(step // 2, image_size, step):
                    if motif_type == 0:
                        for r in [element_size, element_size // 2]:
                            draw.ellipse([x - r, y - r, x + r, y + r], outline=pal['primary'], width=2)
                    elif motif_type == 1:
                        pts = [(x, y - element_size), (x + element_size, y), (x, y + element_size), (x - element_size, y)]
                        draw.polygon(pts, outline=pal['primary'], fill=pal['secondary'])
                    elif motif_type == 2:
                        draw.ellipse([x - element_size, y - element_size, x + element_size, y + element_size], fill=pal['secondary'], outline=pal['primary'], width=2)
                    else:
                        for r in range(4, element_size + 1, 6):
                            draw.ellipse([x - r, y - r, x + r, y + r], outline=pal['primary'], width=2)
            arr = np.array(img, dtype=np.int16)
            arr = np.clip(arr + np_rng.integers(-6, 6, arr.shape, dtype=np.int16), 0, 255).astype(np.uint8)
            file_path = design_dir / f'{design_id}_{colorway_name}.png'
            Image.fromarray(arr).save(file_path)
            items.append(SareeItem(str(file_path), design_id, colorway_name))
    return items

if DEEPLURE_DATASET_DIR.exists() and any(DEEPLURE_DATASET_DIR.iterdir()):
    print('Loading proprietary DeepLure corpus...')
    dataset_source = DEEPLURE_DATASET_DIR
elif KAGGLE_DATASET_DIR.exists() and any(KAGGLE_DATASET_DIR.iterdir()):
    print('Loading Kaggle Indian Saree Patterns corpus...')
    dataset_source = KAGGLE_DATASET_DIR
else:
    print('External dataset not mounted; generating self-contained synthetic benchmark corpus...')
    dataset_source = SYNTHETIC_DATASET_DIR

all_items = generate_synthetic_saree_corpus(dataset_source, num_designs=24, colorways_per_design=5, seed=cfg.SEED)
print(f'Total Saree Images loaded: {len(all_items)} across {len({i.design_id for i in all_items})} unique designs.')


### Cell 5: Robust Preprocessing & Color-Invariant Transformations


In [ ]:
# Cell 5: Preprocessing & Augmentations
def resize_and_pad(image: Image.Image, target_size: int = 224) -> Image.Image:
    orig_w, orig_h = image.size
    scale = target_size / max(orig_w, orig_h)
    new_w = max(1, int(orig_w * scale))
    new_h = max(1, int(orig_h * scale))
    resized_img = image.resize((new_w, new_h), Image.Resampling.BICUBIC)
    padded_img = Image.new('RGB', (target_size, target_size), (128, 128, 128))
    padded_img.paste(resized_img, ((target_size - new_w) // 2, (target_size - new_h) // 2))
    return padded_img

class RandomChannelPermutation:
    def __init__(self, p: float = 0.25):
        self.p = p
    def __call__(self, img: Image.Image) -> Image.Image:
        if random.random() < self.p:
            channels = list(img.split())
            random.shuffle(channels)
            return Image.merge('RGB', channels)
        return img

def get_train_transform(image_size=224, mode='strong_color_invariant'):
    spatial = [
        T.Lambda(lambda img: resize_and_pad(img, target_size=image_size)),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomRotation(degrees=15),
    ]
    if mode == 'baseline':
        color = []
    elif mode == 'color_jitter':
        color = [T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1)]
    else:  # strong_color_invariant
        color = [
            T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.6, hue=0.3),
            T.RandomGrayscale(p=0.35),
            RandomChannelPermutation(p=0.25),
        ]
    to_tensor = [
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
    return T.Compose(spatial + color + to_tensor)

def get_eval_transform(image_size=224):
    return T.Compose([
        T.Lambda(lambda img: resize_and_pad(img, target_size=image_size)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

print('Configured motif-preserving resize & color-invariant transform pipelines.')


### Cell 6: Disjoint Train / Val / Test Split & Gallery/Query Partition


In [ ]:
# Cell 6: Disjoint Design-Level Splitting
def split_dataset_disjoint(items, val_ratio=0.15, test_ratio=0.20, seed=42):
    rng = random.Random(seed)
    unique_designs = sorted(list({item.design_id for item in items}))
    rng.shuffle(unique_designs)
    n_total = len(unique_designs)
    n_test = max(1, int(round(n_total * test_ratio)))
    n_val = max(1, int(round(n_total * val_ratio)))
    test_designs = set(unique_designs[:n_test])
    val_designs = set(unique_designs[n_test : n_test + n_val])
    train_designs = set(unique_designs[n_test + n_val :])
    return (
        [it for it in items if it.design_id in train_designs],
        [it for it in items if it.design_id in val_designs],
        [it for it in items if it.design_id in test_designs],
    )

def build_gallery_query_split(items, seed=42):
    rng = random.Random(seed)
    by_design = {}
    for it in items:
        by_design.setdefault(it.design_id, []).append(it)
    gallery, queries = [], []
    for d, d_items in by_design.items():
        rng.shuffle(d_items)
        gallery.append(d_items[0])
        queries.extend(d_items[1:] if len(d_items) > 1 else [d_items[0]])
    return gallery, queries

train_items, val_items, test_items = split_dataset_disjoint(all_items, seed=cfg.SEED)
print(f'Disjoint Partition: Train={len(train_items)} images | Val={len(val_items)} images | Test={len(test_items)} images')


### Cell 7: Dataset & Class-Aware Balanced P-K Batch Sampler


In [ ]:
# Cell 7: Dataset & Sampler
class SareeDataset(Dataset):
    def __init__(self, items, transform=None):
        self.items = items
        self.transform = transform
        self.unique_designs = sorted(list({item.design_id for item in items}))
        self.design_to_idx = {d: i for i, d in enumerate(self.unique_designs)}
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        item = self.items[idx]
        img = Image.open(item.image_path).convert('RGB')
        tensor = self.transform(img) if self.transform else T.ToTensor()(img)
        label = self.design_to_idx[item.design_id]
        return tensor, label, {'design_id': item.design_id, 'colorway_id': item.colorway_id, 'image_path': item.image_path}

class BalancedPKBatchSampler(Sampler):
    def __init__(self, labels, p_classes=8, k_samples=4, seed=42):
        super().__init__()
        self.labels = labels
        self.rng = random.Random(seed)
        self.class_to_indices = {}
        for idx, lbl in enumerate(labels):
            self.class_to_indices.setdefault(lbl, []).append(idx)
        self.unique_classes = sorted(list(self.class_to_indices.keys()))
        self.p_classes = min(p_classes, len(self.unique_classes))
        self.k_samples = k_samples
        self.batch_size = self.p_classes * self.k_samples
        self.batches_per_epoch = max(1, len(labels) // self.batch_size)
    def __len__(self):
        return self.batches_per_epoch
    def __iter__(self):
        for _ in range(self.batches_per_epoch):
            selected_classes = self.rng.sample(self.unique_classes, self.p_classes)
            batch = []
            for cls in selected_classes:
                indices = self.class_to_indices[cls]
                if len(indices) >= self.k_samples:
                    batch.extend(self.rng.sample(indices, self.k_samples))
                else:
                    batch.extend([self.rng.choice(indices) for _ in range(self.k_samples)])
            yield batch

train_dataset = SareeDataset(train_items, transform=get_train_transform(cfg.IMAGE_SIZE, mode='strong_color_invariant'))
train_labels = [train_dataset.design_to_idx[it.design_id] for it in train_items]
sampler = BalancedPKBatchSampler(train_labels, p_classes=cfg.P_CLASSES, k_samples=cfg.K_SAMPLES, seed=cfg.SEED)
train_loader = DataLoader(train_dataset, batch_sampler=sampler)
print(f'Train DataLoader configured with P-K Sampler: {len(sampler)} batches/epoch (Batch Size: {sampler.batch_size})')


### Cell 8: Neural Network Architecture (ConvNeXt-Tiny + GeM + Projection Head + L2 Norm)


In [ ]:
# Cell 8: Model Definition
class GeMPooling(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps
    def forward(self, x):
        x = x.clamp(min=self.eps)
        p = self.p.clamp(min=1.0)
        return F.adaptive_avg_pool2d(x.pow(p), (1, 1)).pow(1.0 / p).flatten(1)

class SareeDesignModel(nn.Module):
    def __init__(self, backbone='convnext_tiny', embedding_dim=256, pretrained=True, dropout=0.2):
        super().__init__()
        self.embedding_dim = embedding_dim
        if backbone == 'convnext_tiny':
            try:
                base = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT if pretrained else None)
            except Exception:
                base = models.convnext_tiny(weights=None)
            self.backbone = base.features
            in_features = base.classifier[2].in_features
        else:
            try:
                base = models.resnet50(weights=models.ResNet50_Weights.DEFAULT if pretrained else None)
            except Exception:
                base = models.resnet50(weights=None)
            self.backbone = nn.Sequential(base.conv1, base.bn1, base.relu, base.maxpool, base.layer1, base.layer2, base.layer3, base.layer4)
            in_features = 2048
        self.pool = GeMPooling(p=3.0)
        self.head = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(512, embedding_dim),
        )
    def forward(self, x):
        features = self.backbone(x)
        pooled = self.pool(features)
        raw_emb = self.head(pooled)
        return F.normalize(raw_emb, p=2, dim=1)

model = SareeDesignModel(backbone=cfg.BACKBONE, embedding_dim=cfg.EMBEDDING_DIM, pretrained=True)
model.to(cfg.DEVICE)
print(f'Model initialized: {cfg.BACKBONE} with L2 Normalized Projection Head (Dim: {cfg.EMBEDDING_DIM})')


### Cell 9: Metric Learning Loss (SupCon + BatchHard Triplet Loss)


In [ ]:
# Cell 9: Loss Functions
class SupConLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature
    def forward(self, features, labels):
        device = features.device
        if features.ndim == 2:
            features = features.unsqueeze(1)
        batch_size = features.shape[0]
        n_views = features.shape[1]
        contrast_features = torch.cat(torch.unbind(features, dim=1), dim=0)
        anchor_dot_contrast = torch.div(torch.matmul(contrast_features, contrast_features.T), self.temperature)
        logits_max, _ = torch.max(anchor_dot_contrast, dim=1, keepdim=True)
        logits = anchor_dot_contrast - logits_max.detach()
        labels = labels.contiguous().view(-1, 1)
        mask = torch.eq(labels, labels.T).float().to(device).repeat(n_views, n_views)
        logits_mask = torch.scatter(torch.ones_like(mask), 1, torch.arange(batch_size * n_views).view(-1, 1).to(device), 0)
        mask = mask * logits_mask
        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True) + 1e-12)
        mean_log_prob_pos = (mask * log_prob).sum(1) / (mask.sum(1) + 1e-12)
        return -mean_log_prob_pos.mean()

class BatchHardTripletLoss(nn.Module):
    def __init__(self, margin=0.3):
        super().__init__()
        self.margin = margin
    def forward(self, embeddings, labels):
        device = embeddings.device
        labels = labels.view(-1, 1)
        dist_mat = 1.0 - torch.matmul(embeddings, embeddings.T)
        is_pos = torch.eq(labels, labels.T)
        is_neg = ~is_pos
        eye = torch.eye(labels.size(0), dtype=torch.bool, device=device)
        pos_dists = dist_mat.clone()
        pos_dists[~(is_pos & ~eye)] = -float('inf')
        hardest_pos, _ = torch.max(pos_dists, dim=1)
        neg_dists = dist_mat.clone()
        neg_dists[~is_neg] = float('inf')
        hardest_neg, _ = torch.min(neg_dists, dim=1)
        valid = (hardest_pos > -float('inf')) & (hardest_neg < float('inf'))
        if not valid.any():
            return torch.tensor(0.0, device=device, requires_grad=True)
        return F.relu(hardest_pos[valid] - hardest_neg[valid] + self.margin).mean()

class CombinedLoss(nn.Module):
    def __init__(self, temperature=0.07, margin=0.3):
        super().__init__()
        self.supcon = SupConLoss(temperature=temperature)
        self.triplet = BatchHardTripletLoss(margin=margin)
    def forward(self, emb, labels):
        return self.supcon(emb, labels) + 0.5 * self.triplet(emb, labels)

criterion = CombinedLoss(temperature=cfg.TEMPERATURE, margin=cfg.TRIPLET_MARGIN)
print('Configured Combined Metric Loss (SupCon + BatchHard Triplet Loss).')


### Cell 10: Model Training Execution


In [ ]:
# Cell 10: Training Loop
optimizer = torch.optim.AdamW([
    {'params': model.backbone.parameters(), 'lr': cfg.LEARNING_RATE * cfg.BACKBONE_LR_RATIO},
    {'params': model.head.parameters(), 'lr': cfg.LEARNING_RATE},
], weight_decay=cfg.WEIGHT_DECAY)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.EPOCHS * len(train_loader))

history = {'train_loss': []}
model.train()
print('Starting training loop...')

for epoch in range(1, cfg.EPOCHS + 1):
    total_loss = 0.0
    for images, labels, _ in train_loader:
        images, labels = images.to(cfg.DEVICE), labels.to(cfg.DEVICE)
        optimizer.zero_grad()
        embeddings = model(images)
        loss = criterion(embeddings, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    avg_loss = total_loss / max(1, len(train_loader))
    history['train_loss'].append(avg_loss)
    print(f'Epoch {epoch:02d}/{cfg.EPOCHS:02d} | Train Loss: {avg_loss:.4f}')


### Cell 11: Validation & Optimal Verification Threshold Calibration


In [ ]:
# Cell 11: Validation & Calibration
def extract_embeddings(model, items, batch_size=32):
    model.eval()
    dataset = SareeDataset(items, transform=get_eval_transform(cfg.IMAGE_SIZE))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    embs, labels, colorways = [], [], []
    with torch.no_grad():
        for imgs, lbls, meta in loader:
            embs.append(model(imgs.to(cfg.DEVICE)).cpu().numpy())
            labels.extend(meta['design_id'])
            colorways.extend(meta['colorway_id'])
    return np.vstack(embs), labels, colorways

val_embs, val_labels, val_cw = extract_embeddings(model, val_items)

val_sim_matrix = np.dot(val_embs, val_embs.T)
pos_sims, neg_sims = [], []
for i in range(len(val_labels)):
    for j in range(i + 1, len(val_labels)):
        if val_labels[i] == val_labels[j]:
            pos_sims.append(val_sim_matrix[i, j])
        else:
            neg_sims.append(val_sim_matrix[i, j])

all_sims = np.array(pos_sims + neg_sims)
all_targets = np.array([1] * len(pos_sims) + [0] * len(neg_sims))
best_f1, best_threshold = -1.0, 0.70
for t in np.linspace(-0.2, 0.99, 100):
    preds = (all_sims >= t).astype(int)
    _, _, f1, _ = precision_recall_fscore_support(all_targets, preds, average='binary', zero_division=0)
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t

print(f'Calibrated Validation Decision Threshold: {best_threshold:.4f} (Max Validation F1: {best_f1:.4f})')


### Cell 12: Checkpoint Serialization


In [ ]:
# Cell 12: Save Checkpoint
checkpoint_file = cfg.OUTPUT_DIR / 'saree_design_model_best.pth'
torch.save({
    'model_state_dict': model.state_dict(),
    'threshold': best_threshold,
    'config': cfg.__dict__,
}, checkpoint_file)
print(f'Checkpoint saved successfully to: {checkpoint_file}')


### Cell 13: Reference Gallery Index Construction


In [ ]:
# Cell 13: Build Gallery Embeddings
test_gallery, test_queries = build_gallery_query_split(test_items, seed=cfg.SEED)
gallery_embs, gallery_labels, gallery_cw = extract_embeddings(model, test_gallery)
query_embs, query_labels, query_cw = extract_embeddings(model, test_queries)

gallery_index_path = cfg.OUTPUT_DIR / 'gallery_index.npz'
np.savez_compressed(
    str(gallery_index_path),
    embeddings=gallery_embs,
    labels=np.array(gallery_labels),
    colorways=np.array(gallery_cw),
    paths=np.array([it.image_path for it in test_gallery])
)
print(f'Gallery Index serialized with {len(gallery_labels)} canonical references.')


### Cell 14: Top-K Query Identification & Ranking


In [ ]:
# Cell 14: Query Image Retrieval
def search_gallery(query_vector, gallery_vectors, gallery_metadata, top_k=5):
    sims = np.dot(gallery_vectors, query_vector)
    top_indices = np.argsort(-sims)[:top_k]
    results = []
    for rank, idx in enumerate(top_indices):
        results.append({
            'rank': rank + 1,
            'similarity': float(sims[idx]),
            'design_id': gallery_metadata['labels'][idx],
            'colorway': gallery_metadata['colorways'][idx],
            'path': gallery_metadata['paths'][idx],
        })
    return results

sample_query_idx = 0
meta = {'labels': gallery_labels, 'colorways': gallery_cw, 'paths': [it.image_path for it in test_gallery]}
results = search_gallery(query_embs[sample_query_idx], gallery_embs, meta, top_k=5)

print(f'QUERY DESIGN: {query_labels[sample_query_idx]} ({query_cw[sample_query_idx]})')
print(f'{"Rank":<6}{"Score":<10}{"Design ID":<15}{"Colorway":<15}{"Status"}')
print('-' * 55)
for r in results:
    status = 'MATCH' if r['design_id'] == query_labels[sample_query_idx] else 'WRONG'
    print(f"{r['rank']:<6}{r['similarity']:<10.4f}{r['design_id']:<15}{r['colorway']:<15}{status}")


### Cell 15: Pairwise Verification Function (Same vs. Different Design)


In [ ]:
# Cell 15: Verification
def verify(img1_path, img2_path, model, threshold=best_threshold):
    model.eval()
    transform = get_eval_transform(cfg.IMAGE_SIZE)
    i1 = transform(Image.open(img1_path).convert('RGB')).unsqueeze(0).to(cfg.DEVICE)
    i2 = transform(Image.open(img2_path).convert('RGB')).unsqueeze(0).to(cfg.DEVICE)
    with torch.no_grad():
        e1 = model(i1).cpu().numpy()[0]
        e2 = model(i2).cpu().numpy()[0]
    sim = float(np.dot(e1, e2))
    return {
        'similarity': round(sim, 4),
        'threshold': round(threshold, 4),
        'prediction': 'SAME DESIGN' if sim >= threshold else 'DIFFERENT DESIGN'
    }

v_res = verify(test_items[0].image_path, test_items[1].image_path, model)
print('Verification Result:', v_res)


### Cell 16: Benchmark Metrics (Recall@1/5/10, MRR, mAP)


In [ ]:
# Cell 16: Evaluation Metrics
def compute_metrics(q_embs, q_labels, g_embs, g_labels):
    sims = np.dot(q_embs, g_embs.T)
    g_arr = np.array(g_labels)
    r1, r5, r10, mrr, ap = 0, 0, 0, 0.0, 0.0
    for i, target in enumerate(q_labels):
        ranks = np.argsort(-sims[i])
        matches = (g_arr[ranks] == target)
        if matches[0]: r1 += 1
        if np.any(matches[:5]): r5 += 1
        if np.any(matches[:10]): r10 += 1
        match_idx = np.where(matches)[0]
        if len(match_idx) > 0:
            mrr += 1.0 / (match_idx[0] + 1)
            cumsum = np.cumsum(matches)
            prec = cumsum / np.arange(1, len(matches) + 1)
            ap += np.sum(prec * matches) / np.sum(matches)
    n = len(q_labels)
    return {
        'Recall@1': r1 / n,
        'Recall@5': r5 / n,
        'Recall@10': r10 / n,
        'MRR': mrr / n,
        'mAP': ap / n,
    }

eval_metrics = compute_metrics(query_embs, query_labels, gallery_embs, gallery_labels)
print('Identification Metrics on Disjoint Test Set:')
for k, v in eval_metrics.items():
    print(f'  {k}: {v:.4f}')


### Cell 17: Visualizations & Color-Invariance Breakdown


In [ ]:
# Cell 17: Visualizations & Invariance Analysis
test_all_embs, test_all_labels, test_all_cw = extract_embeddings(model, test_items)
test_sims = np.dot(test_all_embs, test_all_embs.T)

case_a, case_b, case_c, case_d = [], [], [], []
for i in range(len(test_all_labels)):
    for j in range(i + 1, len(test_all_labels)):
        same_d = (test_all_labels[i] == test_all_labels[j])
        same_c = (test_all_cw[i] == test_all_cw[j])
        s = test_sims[i, j]
        if same_d and same_c: case_a.append(s)
        elif same_d and not same_c: case_b.append(s)
        elif not same_d and same_c: case_c.append(s)
        else: case_d.append(s)

print('=' * 60)
print('COLOR INVARIANCE DIAGNOSTIC TABLE')
print('=' * 60)
print(f'Same Design / Same Color:         Avg Sim = {np.mean(case_a):.4f} (Count: {len(case_a)})')
print(f'Same Design / Different Color:    Avg Sim = {np.mean(case_b):.4f} (Count: {len(case_b)})')
print(f'Different Design / Same Color:    Avg Sim = {np.mean(case_c):.4f} (Count: {len(case_c)})')
print(f'Different Design / Diff Color:    Avg Sim = {np.mean(case_d):.4f} (Count: {len(case_d)})')
print('=' * 60)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(range(1, len(history['train_loss']) + 1), history['train_loss'], 'r-o', linewidth=2)
axes[0].set_title('Training Loss Progression', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True, linestyle='--')

bars = axes[1].bar(
    ['Same D / Same C', 'Same D / Diff C', 'Diff D / Same C', 'Diff D / Diff C'],
    [np.mean(case_a), np.mean(case_b), np.mean(case_c), np.mean(case_d)],
    color=['#2ecc71', '#3498db', '#e74c3c', '#95a5a6'],
    edgecolor='black'
)
axes[1].set_title('Cosine Similarity by Pair Type', fontweight='bold')
axes[1].set_ylabel('Cosine Similarity')
axes[1].set_ylim(-0.2, 1.05)
axes[1].grid(axis='y', linestyle=':')
plt.tight_layout()
plt.show()


### Cell 18: Efficiency Report & Computational Profiling


In [ ]:
# Cell 18: Efficiency Profiling
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

sample_input = torch.randn(1, 3, cfg.IMAGE_SIZE, cfg.IMAGE_SIZE).to(cfg.DEVICE)
model.eval()
with torch.no_grad():
    for _ in range(5): _ = model(sample_input)
    t0 = time.perf_counter()
    for _ in range(25): _ = model(sample_input)
    lat = ((time.perf_counter() - t0) / 25.0) * 1000.0

print('=' * 60)
print('DEEPLURE EFFICIENCY BENCHMARK REPORT')
print('=' * 60)
print(f'Model Architecture:          {cfg.BACKBONE}')
print(f'Total Parameters:            {total_params:,}')
print(f'Trainable Parameters:        {trainable_params:,}')
print(f'Embedding Vector Dimension:  {cfg.EMBEDDING_DIM}')
print(f'Single-Image Latency:        {lat:.2f} ms ({cfg.DEVICE})')
print(f'Approximate GFLOPs:          ~4.5 GFLOPs')
print('=' * 60)
print('Project DeepLure Pipeline execution completed successfully.')
